## 1. Tree-Based Methods

Import packages

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, roc_curve, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

### (a) Download the APS Failure data

In [ ]:
# Read in csv
aps_failure_train_set = pd.read_csv('../data/aps_failure_training_set.csv', na_values='na', skiprows=20)
aps_failure_test_set = pd.read_csv('../data/aps_failure_test_set.csv', na_values='na', skiprows=20)

# Change class labels to binary
aps_failure_train_set['class'] = aps_failure_train_set['class'].map({'pos':1, 'neg':0})
aps_failure_test_set['class'] = aps_failure_test_set['class'].map({'pos':1, 'neg':0})

# Data exploration print statements
# print(aps_failure_train_set.shape)
# print(aps_failure_test_set.shape)
# print(aps_failure_train_set.dtypes)
# print(aps_failure_test_set.dtypes)
# print(aps_failure_train_set.head())
# print(aps_failure_test_set.head())
# print(aps_failure_train_set['class'].value_counts())
# print(aps_failure_train_set['class'].isnull().sum())

### (b) Data Preparation

#### (i) Research what types of techniques are usually used

##### Answer
* There are many different ways to deal with missing values in our data that fall under the category of removal or imputation based. Since were dealing with so many missing values we should not drop rows or drop columns
* Mean/Median imputation = replace NA with the column's mean or median such that it's preferred when data is skewed
* Mode imputation = replace NA with the most frequent value
* KNN imputation = replace NA with values based on similar rows but requires a lot more computation
* Iterative imputation = we model each feature as a function of other features
* For this dataset I will be doing median imputation because sensor data is often skewed thus using the median would help combat that when compared to the mean being distorted by outliers.

In [ ]:
# Features split for train and test set
X_train = aps_failure_train_set.drop(columns=['class'])
y_train = aps_failure_train_set['class']
X_test = aps_failure_test_set.drop(columns=['class'])
y_test = aps_failure_test_set['class']

# Median imputation
median_imputer = SimpleImputer(strategy='median')
X_train_imputed = median_imputer.fit_transform(X_train)
X_train_imputed = pd.DataFrame(X_train_imputed, columns=X_train.columns) # SimpleImputer returns a numpy array with no column names thus we should return it back to a dataframe
X_test_imputed = median_imputer.transform(X_test)
X_test_imputed = pd.DataFrame(X_test_imputed, columns=X_test.columns)

#### (ii) Calculate the coefficient of variation

In [ ]:
coefficient_of_variation_CV = X_train_imputed.std() / X_train_imputed.mean()
print(coefficient_of_variation_CV.sort_values(ascending=False))
# print(X_train_imputed['cd_000'].nunique())

#### (iii) Plot a correlation matrix

In [ ]:
# Plot creation
plt.figure(figsize=(15,10))
sns.heatmap(X_train_imputed.corr(), cmap='coolwarm', annot=False, vmin=-1, vmax=1)
plt.title('APS Features Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
correlation_matrix = X_train_imputed.corr()
negative_correlation = correlation_matrix[correlation_matrix < 0].stack().sort_values()
print(negative_correlation.head(20))

##### Observations
* There is a strong correlation between features in the same bin especially ba_000 through bt_000
* There are very few weakly negative correlated features that exists in our dataset, however, there are no strongly negative correlated features
* There is a black crosshair line near cc_000 / cd_000 because I found earlier that cd_000 has only 1 unique value throughout the whole column thus that column is constant

#### (iv) Make scatter plots and box plots

In [ ]:
# Floor sqrt(170) = 13
top_13_features = coefficient_of_variation_CV.sort_values(ascending=False).head(13)
# print(top_13_features.index)

# Scatterplot creation
fig, ax = plt.subplots(nrows=4, ncols=4, figsize=(15,10))
ax = ax.flatten()
for i, feature_name in enumerate(top_13_features.index):
    ax[i].scatter(y_train, X_train_imputed[feature_name])
    ax[i].set_xlabel('Class')
    ax[i].set_ylabel(feature_name)
    ax[i].set_title(feature_name)
    ax[i].set_yscale('symlog')

for j in range(len(top_13_features), len(ax)):
    plt.delaxes(ax[j])

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot creation
fig, ax = plt.subplots(nrows=4, ncols=4, figsize=(15,10))
ax = ax.flatten()
for i, feature_name in enumerate(top_13_features.index):
    sns.boxplot(x=y_train, y=X_train_imputed[feature_name], ax=ax[i])
    ax[i].set_xlabel('Class')
    ax[i].set_ylabel(feature_name)
    ax[i].set_title(feature_name)
    ax[i].set_yscale('log')

for j in range(len(top_13_features), len(ax)):
    plt.delaxes(ax[j])

plt.tight_layout()
plt.show()

##### Answer
* No we cannot draw any significance conclusions just based on the scatterplot alone due to the fact that we cannot see the distribution in a scatterplot easily. Furthermore, our data does have a huge class imbalance as stated by the researchers, there are 59000 class 0 and 1000 class 1 thus making it hard to make conclusions because there are just so many more data points part of class 0.
* Lastly, we can't come to a conclusion on feature significance just based on the scatter plot because it only examines one feature at a time. There is a possibility that a combination of those features can form a good predictor.

#### (v) Is this data set imbalanced?

In [ ]:
print(y_train.value_counts())

##### Answer
* Yes, there is a heavy class imbalance such that there are 59000 class 0 (negative) and 1000 class 1 (positive)
* Since the imbalance is so great our model can just predict class 0 for everything and still achieve a high accuracy rate

### (c) Train a random forest

In [ ]:
# Model creation fit
random_forest_imbalanced_classifier = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
random_forest_imbalanced_classifier.fit(X_train_imputed, y_train)

# Model predictions and probability
y_train_predictions = random_forest_imbalanced_classifier.predict(X_train_imputed)
y_train_probability = random_forest_imbalanced_classifier.predict_proba(X_train_imputed)[:,1]
y_test_predictions = random_forest_imbalanced_classifier.predict(X_test_imputed)
y_test_probability = random_forest_imbalanced_classifier.predict_proba(X_test_imputed)[:,1]

In [ ]:
# Training confusion matrix and ROC/AUC creation
rf_train_cm = confusion_matrix(y_train, y_train_predictions)
print(rf_train_cm)
RocCurveDisplay.from_predictions(y_train, y_train_probability)
plt.show()

In [ ]:
# Test confusion matrix and ROC/AUC creation
rf_test_cm = confusion_matrix(y_test, y_test_predictions)
print(rf_test_cm)
RocCurveDisplay.from_predictions(y_test, y_test_probability)
plt.show()

In [ ]:
# Misclassification rate
rf_imbalance_train_misclassification = 1 - accuracy_score(y_train, y_train_predictions)
rf_imbalance_test_misclassification = 1 - accuracy_score(y_test, y_test_predictions)
print(f'Random Forest Imbalance Train Misclassification: {rf_imbalance_train_misclassification}')
print(f'Random Forest Imbalance Test Misclassification: {rf_imbalance_test_misclassification}')

# Out of bag error
print(f'Out of Bag Score: {random_forest_imbalanced_classifier.oob_score_}')
print(f'Out of Bag Error: {1 - random_forest_imbalanced_classifier.oob_score_}')

##### Answer
* The random forest imbalanced test misclassification was 0.76%, whereas, the out of bag error was 0.62% which is close to the test misclassification rate such that it is a reliable generalization error that doesn't require a validation set.

### (d) Research class imbalance in random forest

In [ ]:
# Random forest balanced model creation
random_forest_balanced_classifier = RandomForestClassifier(n_estimators=100, oob_score=True, class_weight='balanced', random_state=42)
random_forest_balanced_classifier.fit(X_train_imputed, y_train)

# Random forest balanced model predictions
y_train_predictions = random_forest_balanced_classifier.predict(X_train_imputed)
y_train_probability = random_forest_balanced_classifier.predict_proba(X_train_imputed)[:, 1]
y_test_predictions = random_forest_balanced_classifier.predict(X_test_imputed)
y_test_probability = random_forest_balanced_classifier.predict_proba(X_test_imputed)[:, 1]

In [ ]:
# Random forest balanced train confusion matrix and AUC/ROC creation
rf_balanced_train_cm = confusion_matrix(y_train, y_train_predictions)
print(rf_balanced_train_cm)
RocCurveDisplay.from_predictions(y_train, y_train_probability)
plt.show()

In [ ]:
# Random forest balanced test confusion matrix and AUC/ROC creation
rf_balanced_test_cm = confusion_matrix(y_test, y_test_predictions)
print(rf_balanced_test_cm)
RocCurveDisplay.from_predictions(y_test, y_test_probability)
plt.show()

In [ ]:
# Misclassification rate
rf_balance_train_misclassification = 1 - accuracy_score(y_train, y_train_predictions)
rf_balance_test_misclassification = 1 - accuracy_score(y_test, y_test_predictions)
print(f'Random Forest Balance Train Misclassification: {rf_balance_train_misclassification}')
print(f'Random Forest Balance Test Misclassification: {rf_balance_test_misclassification}')

# Out of bag error
print(f'Out of Bag Score: {random_forest_balanced_classifier.oob_score_}')
print(f'Out of Bag Error: {1 - random_forest_balanced_classifier.oob_score_}')

##### Answer
* The balanced test misclassification was 1.08%, whereas, the out of bag error was 0.81% thus the out of bag error can be reasonable as a generalization estimate
* After balancing the classes I found that the balanced test misclassification went up actually when compared to the imbalanced, from 0.76% to 1.08% thus balancing was worse than the imbalanced classes
* The test AUC remained the same for both balance and imbalance
* Lastly, the balance classifier encountered more false negative cases when compared to the imbalanced but according to the researchers false negatives in this case are weighed worse than false positives. In conclusion, the balanced classifier was worse than the imbalanced classifier.

### (e) XGBoost and Model Trees

In [ ]:
np.random.seed(42)
# XGBoost classifier creation and fit
alphas = {'reg_alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
xgb_model = XGBClassifier(booster='gblinear', objective='binary:logistic', random_state=42, n_jobs=1)
grid_search_xgb_model = GridSearchCV(xgb_model, alphas, cv=10, scoring='roc_auc')
grid_search_xgb_model.fit(X_train_imputed, y_train)

# Predictions
y_train_predictions = grid_search_xgb_model.predict(X_train_imputed)
y_train_probability = grid_search_xgb_model.predict_proba(X_train_imputed)[:,1]
y_test_predictions = grid_search_xgb_model.predict(X_test_imputed)
y_test_probability = grid_search_xgb_model.predict_proba(X_test_imputed)[:,1]

print(f'Optimal Regularization Term: {grid_search_xgb_model.best_params_['reg_alpha']}')
print(f'Optimal CV AUC: {grid_search_xgb_model.best_score_}')

In [ ]:
# Train confusion matrix and ROC/AUV creation
xgb_imbalanced_train_cm = confusion_matrix(y_train, y_train_predictions)
print(xgb_imbalanced_train_cm)
RocCurveDisplay.from_predictions(y_train, y_train_probability)
plt.show()

In [ ]:
# Test confusion matrix and ROC/AUV creation
xgb_imbalanced_test_cm = confusion_matrix(y_test, y_test_predictions)
print(xgb_imbalanced_test_cm)
RocCurveDisplay.from_predictions(y_test, y_test_probability)
plt.show()

In [ ]:
optimal_xgb = grid_search_xgb_model.best_estimator_ # Uses the optimal regularization term calculated prior
cv_scores = cross_val_score(optimal_xgb, X_train_imputed, y_train, cv=10, scoring='accuracy')
cv_error = 1 - cv_scores.mean()
test_error = 1 - accuracy_score(y_test, y_test_predictions)

print(f'XGB Classifier CV Error: {cv_error}')
print(f'XGB Classifier Test Error: {test_error}')

##### Answer
* The cross validated error was 1.11%, whereas, the test error was 1.23%
* Since they're extremely close we can say that cross validation did give us a good estimate of the generalized performance of the model

### (f) Use SMOTE to pre-process your data

In [ ]:
# Model and pipeline creation
alphas = {'xgb__reg_alpha': [0.001, 0.01, 0.1, 1, 10, 100, 500, 1000]}
smote_xgb_classifier_pipeline = ImbPipeline([('smote', SMOTE(random_state=42)), ('xgb', XGBClassifier(booster='gblinear', objective='binary:logistic', random_state=42))])
grid_search_xgb_SMOTE_model = GridSearchCV(smote_xgb_classifier_pipeline, alphas, cv=10, scoring='roc_auc')
grid_search_xgb_SMOTE_model.fit(X_train_imputed, y_train)

# Predictions
y_train_predictions = grid_search_xgb_SMOTE_model.predict(X_train_imputed)
y_train_probability = grid_search_xgb_SMOTE_model.predict_proba(X_train_imputed)[:,1]
y_test_predictions = grid_search_xgb_SMOTE_model.predict(X_test_imputed)
y_test_probability = grid_search_xgb_SMOTE_model.predict_proba(X_test_imputed)[:,1]

print(f'Optimal Regularization Term: {grid_search_xgb_SMOTE_model.best_params_['xgb__reg_alpha']}')
print(f'Optimal CV AUC: {grid_search_xgb_SMOTE_model.best_score_}')

In [ ]:
# Train confusion matrix and ROC/AUV creation
xgb_SMOTE_train_cm = confusion_matrix(y_train, y_train_predictions)
print(xgb_SMOTE_train_cm)
RocCurveDisplay.from_predictions(y_train, y_train_probability)
plt.show()

In [ ]:
# Test confusion matrix and ROC/AUV creation
xgb_SMOTE_test_cm = confusion_matrix(y_test, y_test_predictions)
print(xgb_SMOTE_test_cm)
RocCurveDisplay.from_predictions(y_test, y_test_probability)
plt.show()

In [ ]:
optimal_xgb_SMOTE = grid_search_xgb_SMOTE_model.best_estimator_ # Uses the optimal regularization term calculated prior
cv_scores = cross_val_score(optimal_xgb_SMOTE, X_train_imputed, y_train, cv=10, scoring='accuracy')
cv_error = 1 - cv_scores.mean()
test_error = 1 - accuracy_score(y_test, y_test_predictions)

print(f'XGB SMOTE Classifier CV Error: {cv_error}')
print(f'XGB SMOTE Classifier Test Error: {test_error}')

##### Answer
* The cross validated error was 2.28%, whereas, the test error was 2.58%
* Since the test and CV error were extremely close we can say that cross validation did give us a good estimate of the generalized performance of the model
* SMOTE XGBoost did perform worse than the non SMOTE XGBoost model, the test error for SMOTE XGBoost was 2.58% and non SMOTE XGBoost was 1.23%
* However, there is a huge caveat for this because the SMOTE XGBoost model had a significant decrease in false negatives of 30 when compared to the false negatives of non SMOTE XGBoost model of 64 in the test set. This is significant because according to the researchers a false negative costed the truck company more when compared to a false positive, so if were purely talking about the cost metric then the SMOTE XGBoost did perform better when compared to the non SMOTE XGBoost model